# Phase 8: End-to-End Evaluation

**Pipeline**: Vietnamese Financial News RAG System — v3  
**Owner**: Member C | **Hardware**: Colab CPU

### Evaluation layers
| Layer | Method | API |
|-------|--------|-----|
| 1a | ROUGE-L (lexical overlap) | None |
| 1b | BERTScore (`bert-base-multilingual-cased`, `lang=vi`) | None |
| 2a | LLM-as-Judge: **Faithfulness** (1-5) | Groq |
| 2b | LLM-as-Judge: **Answer Relevance** (1-5) | Groq |

### Anti-crash mechanisms (re-used from Phase 7)
- **Global Cooldown**: `time.sleep(20)` between every pair of Groq calls  
- **Checkpoint**: parquet saved every 20 rows  
- **Idempotent resume**: completed rows detected and skipped on restart

> ⚠️ Do NOT run LLM-judge calls in a straight loop without cooldown — Groq free tier = 30 RPM.

## Cell 0 — Environment Setup

In [18]:
import os, sys, subprocess
from pathlib import Path

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()
print(f"Runtime: {'Google Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path('/content/rag-vn-finance')
    if not REPO_ROOT.exists():
        subprocess.run(['git', 'clone',
                        'https://github.com/thong7d/rag-vn-finance.git',
                        str(REPO_ROOT)])
        os.system(f'pip install -r "{REPO_ROOT}/requirements.txt" -q')
        os.kill(os.getpid(), 9)
    else:
        print("Repo already exists. Skipping install.")
    from dotenv import load_dotenv
    load_dotenv('/content/drive/MyDrive/rag-vn-finance/.env')
else:
    REPO_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())

print(f"Project root: {REPO_ROOT}")
assert REPO_ROOT.exists(), f"Not found: {REPO_ROOT}"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Cell 0 complete.")

Runtime: Google Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already exists. Skipping install.
Project root: /content/rag-vn-finance
Cell 0 complete.


## Cell 1 — Imports, Config & API Key Check

In [19]:
import time
import json
import pandas as pd
import numpy as np

from src.utils import load_config, resolve_path, ensure_dir, get_env, setup_logger
from src.evaluation import (
    compute_rouge_l,
    compute_bertscore,
    evaluate_faithfulness,
    evaluate_answer_relevance,
    GROQ_COOLDOWN,
    CHECKPOINT_EVERY,
)

logger = setup_logger("Phase8")
config = load_config()

# ── Paths ───────────────────────────────────────────────────────────────────
eval_dir   = resolve_path(config['evaluation'], 'output_dir')
ensure_dir(eval_dir)

gen_results_path   = os.path.join(eval_dir, 'generation_results.parquet')
eval_scores_path   = os.path.join(eval_dir, 'eval_scores.parquet')
final_table_path   = os.path.join(eval_dir, 'final_comparison_table.csv')

assert os.path.exists(gen_results_path), (
    f"generation_results.parquet not found at {gen_results_path}. "
    "Complete Phase 7 first."
)

# ── API key check ────────────────────────────────────────────────────────────
groq_key = get_env('GROQ_API_KEY')
assert groq_key, "GROQ_API_KEY not set. LLM-as-Judge cannot run."
print(f"✅ GROQ_API_KEY found — judge model: {config['evaluation']['groq_model']}")
print(f"   Global Cooldown : {GROQ_COOLDOWN}s between judge calls")
print(f"   Checkpoint every: {CHECKPOINT_EVERY} rows")

✅ GROQ_API_KEY found — judge model: llama-3.3-70b-versatile
   Global Cooldown : 20s between judge calls
   Checkpoint every: 20 rows


## Cell 2 — Load Phase 7 Generation Results

In [20]:
df_gen = pd.read_parquet(gen_results_path)
print(f"Loaded {len(df_gen):,} generation results")
print(f"Columns: {df_gen.columns.tolist()}")

# Normalise column names (Phase 7 may use 'ground_truth' or 'reference_answer')
if 'ground_truth' in df_gen.columns:
    df_gen['reference_answer'] = df_gen['ground_truth']
elif 'reference_answer' not in df_gen.columns:
    raise KeyError("Neither 'ground_truth' nor 'reference_answer' column found.")

# Filter out rows where generation failed entirely
n_before = len(df_gen)
df_gen = df_gen[df_gen['generated_answer'].notna()].copy()
df_gen = df_gen[df_gen['generated_answer'] != '[GENERATION_ERROR]'].copy()
df_gen = df_gen.reset_index(drop=True)
print(f"Valid rows after removing errors: {len(df_gen)} / {n_before}")
df_gen.head(2)

Loaded 197 generation results
Columns: ['question', 'ground_truth', 'retrieved_context', 'generated_answer', 'doc_id', 'strategy', 'method', 'retrieved_chunk_ids']
Valid rows after removing errors: 197 / 197


,question,ground_truth,retrieved_context,generated_answer,doc_id,strategy,method,retrieved_chunk_ids,reference_answer
0,Để tiết kiệm thời gian làm thủ tục tại sân bay...,Hành khách nên làm thủ tục trước chuyến bay qu...,Title: Vietnam Airlines triển khai dịch vụ làm...,Để tiết kiệm thời gian làm thủ tục tại sân bay...,3841852ddb9e0860,fixed_size,Hybrid,"[""0c8c03db6b74adfa_c0001"", ""f595f728ede728d2_c...",Hành khách nên làm thủ tục trước chuyến bay qu...
1,Tình hình tấn công mạng trong ngành ngân hàng ...,"Trong 11 tháng đầu năm 2023, ngành ngân hàng V...",Title: AI và Xác thực Sinh trắc học: Ai là ngư...,"Từ năm 2023 đến quý 1/2024, ngành ngân hàng Vi...",b53f2b7d6b46ec5d,fixed_size,Hybrid,"[""b53f2b7d6b46ec5d_c0000"", ""86ca6e336857e0af_c...","Trong 11 tháng đầu năm 2023, ngành ngân hàng V..."


## Cell 3 — Layer 1a: ROUGE-L

No API calls. Runs locally on CPU in ~30 seconds.

In [21]:
from tqdm import tqdm

rouge_scores = []
for _, row in tqdm(df_gen.iterrows(), total=len(df_gen), desc="ROUGE-L"):
    score = compute_rouge_l(
        prediction=str(row['generated_answer']),
        reference=str(row['reference_answer']),
    )
    rouge_scores.append(score)

df_gen['rouge_l'] = rouge_scores
mean_rouge = np.mean(rouge_scores)
print(f"\nROUGE-L — mean: {mean_rouge:.4f}  "
      f"min: {min(rouge_scores):.4f}  max: {max(rouge_scores):.4f}")

ROUGE-L: 100%|██████████| 197/197 [00:00<00:00, 299.51it/s]


ROUGE-L — mean: 0.5439  min: 0.1474  max: 1.0000


## Cell 4 — Layer 1b: BERTScore

Uses `bert-base-multilingual-cased` with `lang="vi"` — mandatory for Vietnamese.  
Computes the full list in one batch call (~2–5 min on CPU).

In [22]:
print("Computing BERTScore (bert-base-multilingual-cased, lang=vi)...")
print("This may take 2–5 minutes on CPU...")

bert_scores = compute_bertscore(
    predictions=df_gen['generated_answer'].astype(str).tolist(),
    references=df_gen['reference_answer'].astype(str).tolist(),
)

df_gen['bertscore_f1'] = bert_scores
mean_bert = np.mean(bert_scores)
print(f"\nBERTScore F1 — mean: {mean_bert:.4f}  "
      f"min: {min(bert_scores):.4f}  max: {max(bert_scores):.4f}")

Computing BERTScore (bert-base-multilingual-cased, lang=vi)...
This may take 2–5 minutes on CPU...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



BERTScore F1 — mean: 0.8264  min: 0.6427  max: 1.0000


## Cell 5 — Layer 2: LLM-as-Judge (Faithfulness + Answer Relevance)

**Anti-crash design** (mandatory):
- Each row makes exactly **2 Groq API calls** (Faithfulness, then Answer Relevance)
- A **`time.sleep(20)`** Global Cooldown fires between every pair of calls
- Results are **checkpointed every 20 rows** to `eval_scores.parquet`
- If the session crashes, re-running this cell resumes from the last checkpoint

> ⏱️ Expected time: ~197 rows × 40 s/row ≈ **~2.2 hours**. Leave the tab open.

In [23]:
import os
import time
import pandas as pd
from tqdm import tqdm

# ── Resume from checkpoint ────────────────────────────────────────────────────
if os.path.exists(eval_scores_path):
    df_scores_existing = pd.read_parquet(eval_scores_path)

    # BỘ LỌC TỰ ĐỘNG VÁ LỖI: Chỉ coi là "đã hoàn thành" nếu điểm số > 0
    # Gạt bỏ toàn bộ các dòng bị lỗi điểm 0 từ các lần chạy trước để ép chạy lại
    valid_rows = df_scores_existing[(df_scores_existing['faithfulness_score'] > 0) &
                                    (df_scores_existing['answer_relevance_score'] > 0)]

    if 'original_index' in valid_rows.columns:
        completed_indices = set(valid_rows['original_index'].tolist())
    else:
        completed_indices = set(valid_rows.index.tolist())

    all_score_rows = valid_rows.to_dict('records')
    print(f"🔄 Resuming — {len(all_score_rows)} valid rows already evaluated. Skipping them.")

    if len(df_scores_existing) > len(valid_rows):
        print(f"🧹 Đã tự động loại bỏ {len(df_scores_existing) - len(valid_rows)} dòng lỗi (điểm 0) để chấm lại.")
else:
    completed_indices = set()
    all_score_rows    = []
    print("🚀 Starting fresh LLM-as-Judge evaluation...")

remaining = len(df_gen) - len(completed_indices)
print(f"   Rows to evaluate: {remaining}")
print(f"   Estimated time  : ~{remaining * (GROQ_COOLDOWN * 2 + 5) // 60} minutes\n")

# Cờ báo hiệu lỗi để dừng vòng lặp
execution_paused = False

# ── Evaluation loop ───────────────────────────────────────────────────────────
for orig_idx, row in tqdm(df_gen.iterrows(), total=len(df_gen), desc="LLM Judge"):

    if orig_idx in completed_indices:
        continue

    question          = str(row['question'])
    generated_answer  = str(row['generated_answer'])
    retrieved_context = str(row.get('retrieved_context', ''))
    reference_answer  = str(row['reference_answer'])

    try:
        # ── Call 1: Faithfulness ─────────────────────────────────────────────────
        faith = evaluate_faithfulness(
            context=retrieved_context,
            answer=generated_answer,
        )

        if isinstance(faith, dict) and faith.get('score', 1) == 0:
            logger.error(f"Error in Faithfulness (Row {orig_idx}): {faith.get('reasoning')}")
            execution_paused = True
            break # DỪNG NGAY, không lưu dòng lỗi

        # ── Global Cooldown ──────────────────────────────────────────────────────
        logger.info(f"Row {orig_idx}: Faithfulness={faith['score']} — sleeping {GROQ_COOLDOWN}s")
        time.sleep(GROQ_COOLDOWN)

        # ── Call 2: Answer Relevance ─────────────────────────────────────────────
        relevance = evaluate_answer_relevance(
            question=question,
            answer=generated_answer,
        )

        if isinstance(relevance, dict) and relevance.get('score', 1) == 0:
            logger.error(f"Error in Relevance (Row {orig_idx}): {relevance.get('reasoning')}")
            execution_paused = True
            break # DỪNG NGAY, không lưu dòng lỗi

        logger.info(f"Row {orig_idx}: Relevance={relevance['score']}")

    except Exception as e:
        # Bắt TOÀN BỘ lỗi (thiếu API Key, Rate Limit, mất mạng) và thoát ngay
        logger.error(f"Critical Error at row {orig_idx}: {e}")
        execution_paused = True
        break

    # ── Accumulate result (Chỉ chạy đến đây nếu không bị Break) ────────────────
    all_score_rows.append({
        'original_index':            orig_idx,
        'question':                  question,
        'generated_answer':          generated_answer,
        'reference_answer':          reference_answer,
        'rouge_l':                   row.get('rouge_l', 0.0),
        'bertscore_f1':              row.get('bertscore_f1', 0.0),
        'faithfulness_score':        faith.get('score', 0),
        'faithfulness_reasoning':    faith.get('reasoning', ''),
        'answer_relevance_score':    relevance.get('score', 0),
        'answer_relevance_reasoning':relevance.get('reasoning', ''),
        'strategy':                  row.get('strategy', ''),
        'method':                    row.get('method', ''),
        'doc_id':                    row.get('doc_id', ''),
    })

    # ── Checkpoint every CHECKPOINT_EVERY rows ───────────────────────────────
    if len(all_score_rows) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(all_score_rows).to_parquet(eval_scores_path, index=False)
        logger.info(f"[Checkpoint] {len(all_score_rows)} rows saved to {eval_scores_path}")

# Final save after loop completes (or breaks early)
if all_score_rows:
    df_eval_scores = pd.DataFrame(all_score_rows)
    df_eval_scores.to_parquet(eval_scores_path, index=False)

    if execution_paused:
        print(f"\n⚠️ EXECUTION PAUSED: Hệ thống gặp lỗi. {len(df_eval_scores)} dòng hợp lệ đã được lưu.")
        print("   -> Lần chạy tiếp theo sẽ tự động bắt đầu từ câu chưa được đánh giá.")
    else:
        print(f"\n✅ LLM-as-Judge complete — {len(df_eval_scores)} rows evaluated.")
        print(f"   Faithfulness    mean: {df_eval_scores['faithfulness_score'].mean():.2f}/5")
        print(f"   Answer Relevance mean: {df_eval_scores['answer_relevance_score'].mean():.2f}/5")
else:
    print("No rows processed.")

🔄 Resuming — 193 valid rows already evaluated. Skipping them.
   Rows to evaluate: 4
   Estimated time  : ~3 minutes



LLM Judge:   0%|          | 0/197 [00:00<?, ?it/s][2026-05-15 04:52:08] [INFO] Phase8: Row 193: Faithfulness=5 — sleeping 20s
INFO:Phase8:Row 193: Faithfulness=5 — sleeping 20s
[2026-05-15 04:52:28] [INFO] Phase8: Row 193: Relevance=5
INFO:Phase8:Row 193: Relevance=5
LLM Judge:  98%|█████████▊| 194/197 [00:22<00:00,  8.58it/s][2026-05-15 04:52:30] [INFO] Phase8: Row 194: Faithfulness=5 — sleeping 20s
INFO:Phase8:Row 194: Faithfulness=5 — sleeping 20s
[2026-05-15 04:52:51] [INFO] Phase8: Row 194: Relevance=5
INFO:Phase8:Row 194: Relevance=5
LLM Judge:  99%|█████████▉| 195/197 [00:45<00:00,  3.58it/s][2026-05-15 04:52:52] [INFO] Phase8: Row 195: Faithfulness=5 — sleeping 20s
INFO:Phase8:Row 195: Faithfulness=5 — sleeping 20s
[2026-05-15 04:53:13] [INFO] Phase8: Row 195: Relevance=5
INFO:Phase8:Row 195: Relevance=5
LLM Judge:  99%|█████████▉| 196/197 [01:07<00:00,  1.97it/s][2026-05-15 04:53:14] [INFO] Phase8: Row 196: Faithfulness=5 — sleeping 20s
INFO:Phase8:Row 196: Faithfulness=5 — sl


✅ LLM-as-Judge complete — 197 rows evaluated.
   Faithfulness    mean: 4.49/5
   Answer Relevance mean: 4.93/5


## Cell 6 — Aggregate Metrics per Config (Strategy × Method)

In [24]:
# Load the full scores table (handles case where session restarted after Cell 5)
df_scores = pd.read_parquet(eval_scores_path)
print(f"Loaded {len(df_scores)} scored rows")

# Numeric metric columns
METRIC_COLS = [
    'rouge_l', 'bertscore_f1',
    'faithfulness_score', 'answer_relevance_score',
]

# Group by Strategy × Method and compute mean
df_agg = (
    df_scores
    .groupby(['strategy', 'method'])[METRIC_COLS]
    .agg(['mean', 'std'])
    .round(4)
)

# Flatten multi-level column index
df_agg.columns = ['_'.join(c) for c in df_agg.columns]
df_agg = df_agg.reset_index()
print("\nPer-config aggregation:")
print(df_agg.to_string(index=False))

Loaded 197 scored rows

Per-config aggregation:
  strategy method  rouge_l_mean  rouge_l_std  bertscore_f1_mean  bertscore_f1_std  faithfulness_score_mean  faithfulness_score_std  answer_relevance_score_mean  answer_relevance_score_std
fixed_size Hybrid        0.5439       0.1765             0.4837            0.4092                   4.4873                  1.1231                        4.934                      0.4956


## Cell 7 — Merge with Retrieval Metrics & Save Final Comparison Table

In [25]:
benchmark_path = os.path.join(eval_dir, 'retrieval_benchmark.csv')

if os.path.exists(benchmark_path):
    df_retrieval = pd.read_csv(benchmark_path)
    # Normalise column names to lowercase for merge
    df_retrieval = df_retrieval.rename(columns={'Strategy': 'strategy', 'Method': 'method'})
    df_final = df_agg.merge(df_retrieval, on=['strategy', 'method'], how='left')
    print("Merged with retrieval_benchmark.csv")
else:
    df_final = df_agg.copy()
    print("⚠️  retrieval_benchmark.csv not found — final table contains only generation metrics.")

# Save final comparison table
df_final.to_csv(final_table_path, index=False)
print(f"\n✅ Final comparison table saved: {final_table_path}")
print(f"   Rows: {len(df_final)}  Columns: {df_final.columns.tolist()}")
df_final

Merged with retrieval_benchmark.csv

✅ Final comparison table saved: /content/drive/MyDrive/rag-vn-finance/evaluation/final_comparison_table.csv
   Rows: 1  Columns: ['strategy', 'method', 'rouge_l_mean', 'rouge_l_std', 'bertscore_f1_mean', 'bertscore_f1_std', 'faithfulness_score_mean', 'faithfulness_score_std', 'answer_relevance_score_mean', 'answer_relevance_score_std', 'Precision@10', 'Recall@10', 'MRR', 'NDCG@10']


,strategy,method,rouge_l_mean,rouge_l_std,bertscore_f1_mean,bertscore_f1_std,faithfulness_score_mean,faithfulness_score_std,answer_relevance_score_mean,answer_relevance_score_std,Precision@10,Recall@10,MRR,NDCG@10
0,fixed_size,Hybrid,0.5439,0.1765,0.4837,0.4092,4.4873,1.1231,4.934,0.4956,0.306612,0.95117,0.855406,0.840344


## Cell 8 — Best Configuration & Score Summary

In [26]:
df_final = pd.read_csv(final_table_path)

# Rank by composite score: average of normalised MRR, ROUGE-L, BERTScore,
# Faithfulness, and Answer Relevance (all scaled 0-1)
score_cols_available = [c for c in [
    'MRR', 'rouge_l_mean', 'bertscore_f1_mean',
    'faithfulness_score_mean', 'answer_relevance_score_mean'
] if c in df_final.columns]

if score_cols_available:
    # Normalise each column to [0,1] then average
    df_norm = df_final[score_cols_available].copy()
    for col in score_cols_available:
        col_range = df_norm[col].max() - df_norm[col].min()
        df_norm[col] = (df_norm[col] - df_norm[col].min()) / (col_range if col_range > 0 else 1)
    df_final['composite_score'] = df_norm.mean(axis=1).round(4)
    best_idx = df_final['composite_score'].idxmax()
    best_row = df_final.loc[best_idx]
    print("=" * 60)
    print("BEST CONFIGURATION (by composite score)")
    print("=" * 60)
    print(f"  Strategy         : {best_row['strategy']}")
    print(f"  Method           : {best_row['method']}")
    print(f"  Composite Score  : {best_row['composite_score']:.4f}")
    for col in score_cols_available:
        print(f"  {col:30s}: {best_row[col]:.4f}")

print("\n" + "=" * 60)
print("ALL CONFIGURATIONS")
print("=" * 60)
print(df_final[['strategy', 'method'] + score_cols_available].to_string(index=False))

print(f"\n✅ Phase 8 complete. Confirm 'Xong' before proceeding to Phase 9.")

BEST CONFIGURATION (by composite score)
  Strategy         : fixed_size
  Method           : Hybrid
  Composite Score  : 0.0000
  MRR                           : 0.8554
  rouge_l_mean                  : 0.5439
  bertscore_f1_mean             : 0.4837
  faithfulness_score_mean       : 4.4873
  answer_relevance_score_mean   : 4.9340

ALL CONFIGURATIONS
  strategy method      MRR  rouge_l_mean  bertscore_f1_mean  faithfulness_score_mean  answer_relevance_score_mean
fixed_size Hybrid 0.855406        0.5439             0.4837                   4.4873                        4.934

✅ Phase 8 complete. Confirm 'Xong' before proceeding to Phase 9.


## Cell 9 — Error Analysis Sample

Display the 5 rows with the lowest Faithfulness score for manual inspection.

In [27]:
df_scores = pd.read_parquet(eval_scores_path)

# Bottom 5 Faithfulness rows
bottom5 = df_scores.nsmallest(5, 'faithfulness_score')[
    ['question', 'generated_answer', 'faithfulness_score',
     'faithfulness_reasoning', 'answer_relevance_score']
]

print("=== 5 Lowest-Faithfulness Answers ===")
for _, row in bottom5.iterrows():
    print(f"\nFaithfulness={row['faithfulness_score']}  "
          f"Relevance={row['answer_relevance_score']}")
    print(f"Q  : {row['question'][:120]}")
    print(f"GEN: {str(row['generated_answer'])[:200]}")
    print(f"WHY: {row['faithfulness_reasoning']}")

=== 5 Lowest-Faithfulness Answers ===

Faithfulness=1  Relevance=5
Q  : Tiềm năng mở rộng của Tesla trong những năm tới là gì?
GEN: Theo đoạn 3, Tesla đặt mục tiêu đạt sản lượng hàng năm 20 triệu xe vào năm 2030. Điều này cho thấy tiềm năng mở rộng lớn của Tesla trong những năm tới. Ngoài ra, đoạn 1 cũng đề cập đến việc Tesla sẽ c
WHY: The generated answer contains fabricated facts, such as Tesla's goal of producing 20 million vehicles by 2030, which is not mentioned in the retrieved context.

Faithfulness=1  Relevance=5
Q  : HNX-Index phiên 23/11 có biến động gì và đặc điểm nổi bật?
GEN: Trong phiên giao dịch ngày 23/11, HNX-Index trải qua những nhịp rung lắc và liên tục đảo chiều, kết phiên giảm 0,28 điểm (-0,27%) xuống 104,27 điểm. Đặc điểm nổi bật là dù thị trường chung chịu áp lực
WHY: The generated answer contains fabricated facts, such as the HNX-Index decrease and VCG's price, which are not present in the retrieved context.

Faithfulness=1  Relevance=5
Q  : Cơ cấu sử dụng đất h